# 🤖 Grant Agent — Complete Colab Runner

| Cell | What it does | Run? |
|------|-------------|------|
| **1** | Mount Drive, clone repo, install deps | ✅ Always |
| **2** | **Your email + up to 3 Gemini API keys** | ✅ Always |
| **3** | Start server + get your public URL | ✅ Always |
| **4** | Save your email to the org profile | Once |
| **5** | Browser bot on a real grant site | Optional |
| **6** | Scrape a new grant from any URL | Optional |
| **7** | Deploy to Google Cloud Run (24/7) | Optional |

Data lives on Google Drive — re-run Cells 1→2→3 after any session reset.

In [ ]:
# ==============================================================
# CELL 1 — Mount Google Drive & Install Everything
# First run: ~4-5 min. Subsequent runs: ~30 sec.
# ==============================================================

from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys, shutil

DATA_DIR = '/content/drive/MyDrive/GrantAgentData'
for sub in ['db', 'uploads', 'screenshots', 'logs']:
    os.makedirs(f'{DATA_DIR}/{sub}', exist_ok=True)
print(f'✅ Drive folders ready: {DATA_DIR}')

REPO_DIR = '/content/grant-agent'
REPO_URL = 'https://github.com/ademikelz19-hub/cookie-project.git'
BRANCH   = 'feature/grant-agent'

if os.path.exists(f'{REPO_DIR}/.git'):
    print('🔄 Syncing latest code from GitHub...')
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'reset', '--hard', f'origin/{BRANCH}'], cwd=REPO_DIR, check=True)
    print('✅ Code updated to latest commit')
else:
    print('📥 Cloning repository...')
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)

print('\n📦 Installing API dependencies...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '-r', f'{REPO_DIR}/grant-agent-api/requirements.txt'], check=True)

print('📦 Installing worker dependencies...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '-r', f'{REPO_DIR}/grant-agent-worker/requirements.txt'], check=True)

print('🎭 Installing Playwright Chromium...')
subprocess.run([sys.executable, '-m', 'playwright', 'install', 'chromium',
                '--with-deps'], check=True)

if not os.path.exists('/usr/local/bin/cloudflared'):
    print('🌐 Installing cloudflared...')
    subprocess.run(['bash', '-c',
        'wget -q https://github.com/cloudflare/cloudflared/releases/latest/'
        'download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared '
        '&& chmod +x /usr/local/bin/cloudflared'], check=True)

print('\n✅ CELL 1 COMPLETE — run Cell 2 next')


In [ ]:
# ==============================================================
# CELL 2 — YOUR CONFIGURATION
#
# ✏️  EDIT THE LINES BELOW, THEN CLICK ▶
#
# YOUR_EMAIL
#   Your real email. The browser bot fills this on grant forms.
#
# GEMINI_API_KEY_1 / _2 / _3
#   Get free keys at: https://aistudio.google.com/app/apikey
#
#   The system tries keys in order. If Key 1 hits a rate-limit
#   (429 quota error) it automatically switches to Key 2, then
#   Key 3 — without stopping or asking you.
#
#   Each key is also tried across 3 model tiers:
#     gemini-3.8-flash → gemini-3.5-flash → gemini-2.5-flash
#
#   Full matrix: 3 keys × 3 models = up to 9 attempts before fail.
#
#   Only have one key? Leave KEY_2 and KEY_3 as empty strings ''.
#
# GCP_PROJECT_ID — only needed for Cell 7 (Cloud Run deploy).
# ==============================================================

# ── ✏️  EDIT THESE ─────────────────────────────────────────────

YOUR_EMAIL = 'your.real.email@example.com'

GEMINI_API_KEY_1 = ''   # ← paste your first Gemini key here
GEMINI_API_KEY_2 = ''   # ← paste second key (optional fallback)
GEMINI_API_KEY_3 = ''   # ← paste third key  (optional fallback)

GCP_PROJECT_ID = 'grant-agent-gcp'   # only for Cell 7

# ── Auto-set — no need to touch these ─────────────────────────
import secrets
API_PORT   = 8000
GCP_REGION = 'us-central1'
JWT_SECRET = secrets.token_hex(32)

# ── Confirm ────────────────────────────────────────────────────
def _mask(k):
    if not k: return '(not set)'
    return f'{k[:8]}...{k[-4:]}' if len(k) > 16 else k

active_keys = [k for k in [GEMINI_API_KEY_1, GEMINI_API_KEY_2, GEMINI_API_KEY_3] if k]

print('⚙️  Configuration:')
print(f'   Email        : {YOUR_EMAIL}')
print(f'   API Key 1    : {_mask(GEMINI_API_KEY_1)}')
print(f'   API Key 2    : {_mask(GEMINI_API_KEY_2)}')
print(f'   API Key 3    : {_mask(GEMINI_API_KEY_3)}')
print(f'   Active keys  : {len(active_keys)} / 3')
print(f'   GCP project  : {GCP_PROJECT_ID}')
print()

warns = []
if YOUR_EMAIL == 'your.real.email@example.com':
    warns.append('⚠️  YOUR_EMAIL is still the placeholder — update it before running Cell 4')
if not active_keys:
    warns.append('⚠️  No API keys set — AI will run in offline demo mode')
    warns.append('   Get a free key: https://aistudio.google.com/app/apikey')
for w in warns:
    print(w)

if not warns:
    print('✅ Ready — run Cell 3 to start the server')

In [ ]:
# ==============================================================
# CELL 3 — Start Grant Agent + Get Public URL
#
# What this cell does:
#   1. Stops any old running uvicorn process on port 8000
#   2. Restores your SQLite database from Drive (if available)
#      to fast local disk /content/grant_agent.db to avoid
#      Google Drive FUSE file locking errors
#   3. Starts the FastAPI backend with uvicorn
#   4. Opens a public HTTPS tunnel via cloudflared
#   5. Starts a continuous background sync that backs up the DB
#      to Google Drive every 30 seconds
#
# KEEP THIS CELL RUNNING while using Grant Agent.
# ==============================================================

import os, subprocess, threading, time, re, sys, shutil
import urllib.request

REPO_DIR    = '/content/grant-agent'
DATA_DIR    = '/content/drive/MyDrive/GrantAgentData'
api_path    = f'{REPO_DIR}/grant-agent-api'
worker_path = f'{REPO_DIR}/grant-agent-worker'

# 1. Kill any existing uvicorn processes to free the port
subprocess.run(['pkill', '-9', '-f', 'uvicorn'], capture_output=True)
time.sleep(1)

# 2. Database Sync: Use local ext4 disk for SQLite (avoids FUSE locking crashes)
LOCAL_DB = '/content/grant_agent.db'
DRIVE_DB = f'{DATA_DIR}/db/grant_agent.db'
if os.path.exists(DRIVE_DB) and os.path.getsize(DRIVE_DB) > 0 and not os.path.exists(LOCAL_DB):
    shutil.copy2(DRIVE_DB, LOCAL_DB)
    print(f'📦 Restored existing database from Google Drive ({os.path.getsize(LOCAL_DB):,} bytes)')

# 3. Write environment configuration
env_text = f"""# Grant Agent — Colab .env
DATABASE_URL=sqlite:///{LOCAL_DB}
SECRET_KEY={JWT_SECRET}
ALGORITHM=HS256
ACCESS_TOKEN_EXPIRE_MINUTES=10080
GEMINI_API_KEY_1={GEMINI_API_KEY_1}
GEMINI_API_KEY_2={GEMINI_API_KEY_2}
GEMINI_API_KEY_3={GEMINI_API_KEY_3}
GEMINI_API_KEY={GEMINI_API_KEY_1}
PRIMARY_GEMINI_MODEL=gemini-3.8-flash
FALLBACK_GEMINI_MODEL=gemini-3.5-flash
SECONDARY_FALLBACK_GEMINI_MODEL=gemini-2.5-flash
GEMINI_MAX_RETRIES=3
GEMINI_RETRY_BASE_SECONDS=2
GEMINI_REQUEST_TIMEOUT_SECONDS=45
ENVIRONMENT=colab
PROJECT_NAME=Grant Agent
LOG_LEVEL=INFO
UPLOAD_DIR={DATA_DIR}/uploads
SCREENSHOTS_DIR={DATA_DIR}/screenshots
CORS_ORIGINS=["*"]
"""

for target_dir in [REPO_DIR, api_path]:
    with open(f'{target_dir}/.env', 'w') as f:
        f.write(env_text)
print('✅ .env written to repository and API directory')

# 4. Prepare environment variables for child process
env = os.environ.copy()
env['PYTHONPATH'] = f'{api_path}:{worker_path}'
env['DATABASE_URL'] = f'sqlite:///{LOCAL_DB}'
env['SECRET_KEY'] = JWT_SECRET
env['GEMINI_API_KEY_1'] = GEMINI_API_KEY_1
env['GEMINI_API_KEY_2'] = GEMINI_API_KEY_2
env['GEMINI_API_KEY_3'] = GEMINI_API_KEY_3
env['GEMINI_API_KEY'] = GEMINI_API_KEY_1
env['PRIMARY_GEMINI_MODEL'] = 'gemini-3.8-flash'
env['FALLBACK_GEMINI_MODEL'] = 'gemini-3.5-flash'
env['SECONDARY_FALLBACK_GEMINI_MODEL'] = 'gemini-2.5-flash'

for p in [api_path, worker_path]:
    if p not in sys.path:
        sys.path.insert(0, p)

log_path = f'{DATA_DIR}/logs/api_server.log'
log_file = open(log_path, 'w')
print(f'🚀 Starting Grant Agent server on port {API_PORT}...')
server_proc = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'app.main:app',
     '--host', '0.0.0.0', '--port', str(API_PORT), '--log-level', 'info'],
    cwd=api_path, env=env, stdout=log_file, stderr=subprocess.STDOUT
)

# 5. Wait and test /health endpoint with automatic diagnostic logging on error
server_up = False
for attempt in range(20):
    time.sleep(2)
    if server_proc.poll() is not None:
        print(f'⚠️ Server process exited prematurely with code {server_proc.returncode}')
        break
    try:
        with urllib.request.urlopen(f'http://localhost:{API_PORT}/health', timeout=2) as resp:
            if resp.status == 200:
                server_up = True
                print(f'✅ Server is up and healthy! (attempt {attempt+1})')
                break
    except Exception:
        pass

if not server_up:
    log_file.flush()
    print(f'\n❌ Server failed to start on port {API_PORT}.')
    print('='*65)
    print('📜 EXACT SERVER ERROR LOG:')
    print('='*65)
    try:
        with open(log_path, 'r') as lf:
            lines = lf.readlines()
            for l in lines[-35:]:
                print(l.rstrip())
    except Exception as e:
        print('Could not read log file:', e)
    print('='*65)
    raise RuntimeError('Server startup failed — see traceback above')

# 6. Start cloudflared public tunnel
tunnel_url   = None
tunnel_lines = []

def _read_tunnel(proc):
    global tunnel_url
    for line in iter(proc.stdout.readline, b''):
        s = line.decode('utf-8', errors='ignore').strip()
        tunnel_lines.append(s)
        m = re.search(r'https://[\w-]+\.trycloudflare\.com', s)
        if m and not tunnel_url:
            tunnel_url = m.group(0)

tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', f'http://localhost:{API_PORT}'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
threading.Thread(target=_read_tunnel, args=(tunnel_proc,), daemon=True).start()

print('⏳ Connecting public HTTPS tunnel...')
for _ in range(30):
    if tunnel_url: break
    time.sleep(2)

print()
print('='*62)
if tunnel_url:
    print('🌐  GRANT AGENT IS LIVE:')
    print(f'\n      {tunnel_url}')
    print(f'\n  📖  API docs : {tunnel_url}/docs')
    print(f'  ❤️   Health  : {tunnel_url}/health')
else:
    print('⚠️  Tunnel URL could not be detected. Check tunnel_lines.')
print()
print('  🔑  Login: demo@grantagent.ai / password123')
print('  📁  Database is continuously backed up to Google Drive')
print('='*62)

# 7. Background thread: Keep-alive + continuous Google Drive DB backup every 30s
def _background_worker():
    while True:
        try:
            urllib.request.urlopen(f'http://localhost:{API_PORT}/health', timeout=5)
        except:
            pass
        # Auto-backup database to Google Drive
        try:
            if os.path.exists(LOCAL_DB):
                shutil.copy2(LOCAL_DB, DRIVE_DB)
        except Exception:
            pass
        time.sleep(30)

threading.Thread(target=_background_worker, daemon=True).start()
print('\n💓 Keep-alive & Google Drive database auto-sync active (every 30s)')


In [ ]:
# ==============================================================
# CELL 4 — Save YOUR Email to the Organisation Profile
#
# Run once after Cell 3. Email is stored in the Drive database
# and persists across sessions. The browser bot reads it and
# fills it on real grant application forms automatically.
# ==============================================================

import urllib.request, urllib.parse, json

BASE = f'http://localhost:{API_PORT}/api/v1'

login_data = urllib.parse.urlencode(
    {'username': 'demo@grantagent.ai', 'password': 'password123'}).encode()
with urllib.request.urlopen(
        urllib.request.Request(f'{BASE}/auth/login', data=login_data)) as r:
    TOKEN = json.loads(r.read())['access_token']
print('✅ Logged in')

H = {'Authorization': f'Bearer {TOKEN}', 'Content-Type': 'application/json'}

with urllib.request.urlopen(
        urllib.request.Request(f'{BASE}/organisations', headers=H)) as r:
    orgs = json.loads(r.read())

print(f'📋 {len(orgs)} organisation(s):')
for i, o in enumerate(orgs):
    print(f'   [{i}] {o["organisation_name"]} — email: {o.get("email", "(none)")}')

print(f'\n🔄 Updating to: {YOUR_EMAIL}')
skip = {'id','owner_id','created_at','updated_at','founders','team_members','metrics','documents'}
for org in orgs:
    payload = {k: v for k, v in org.items() if k not in skip}
    payload['email'] = YOUR_EMAIL
    req = urllib.request.Request(
        f'{BASE}/organisations/{org["id"]}',
        data=json.dumps(payload).encode(), headers=H, method='PUT')
    with urllib.request.urlopen(req) as r:
        u = json.loads(r.read())
    print(f'   ✅ {u["organisation_name"]} → {u["email"]}')

print(f'\n🎯 Done. Browser bot will fill "{YOUR_EMAIL}" on all grant forms.')
print('   (Saved to Drive — no need to run this cell again)')

In [ ]:
# ==============================================================
# CELL 5 — Browser Automation on a Real Grant Website
#
# Playwright opens the URL, scans the form, fills every field
# it can (including your email), stops at CAPTCHA/OTP and
# saves a screenshot. Never submits without your review.
#
# Change TEST_GRANT_URL to any real grant application page.
# ==============================================================

import urllib.request, urllib.parse, json, time

BASE = f'http://localhost:{API_PORT}/api/v1'

# ── ✏️  EDIT THESE ─────────────────────────────────────────────
TEST_GRANT_URL = 'https://www.tonyelumelufoundation.org/apply'
ORG_INDEX = 0   # 0 = Lioris (startup), 1 = Princess Sara Foundation (nonprofit)
# ──────────────────────────────────────────────────────────────

login_data = urllib.parse.urlencode(
    {'username': 'demo@grantagent.ai', 'password': 'password123'}).encode()
with urllib.request.urlopen(
        urllib.request.Request(f'{BASE}/auth/login', data=login_data)) as r:
    TOKEN = json.loads(r.read())['access_token']
H = {'Authorization': f'Bearer {TOKEN}', 'Content-Type': 'application/json'}

with urllib.request.urlopen(urllib.request.Request(f'{BASE}/organisations', headers=H)) as r:
    orgs = json.loads(r.read())
org = orgs[ORG_INDEX]
print(f'🏢 {org["organisation_name"]} (email: {org.get("email")})')

with urllib.request.urlopen(urllib.request.Request(f'{BASE}/grants', headers=H)) as r:
    grants = json.loads(r.read())
if not grants:
    print('No grants — run Cell 6 first to scrape a grant URL.')
    raise SystemExit(0)
print(f'📄 {grants[0]["grant_name"]}')

app_data = json.dumps({'org_id': org['id'], 'grant_id': grants[0]['id']}).encode()
with urllib.request.urlopen(
        urllib.request.Request(f'{BASE}/applications', data=app_data, headers=H)) as r:
    application = json.loads(r.read())
APP_ID = application['id']
print(f'📝 Application: {APP_ID}')

print(f'\n🤖 Launching browser → {TEST_GRANT_URL}')
bp = json.dumps({'application_id': APP_ID, 'target_url': TEST_GRANT_URL}).encode()
with urllib.request.urlopen(
        urllib.request.Request(f'{BASE}/browser/start', data=bp, headers=H)) as r:
    session = json.loads(r.read())
SID = session.get('id')
print(f'   Session: {SID}')
print('⏳ Polling every 5 s (max 2 min)...')

result = {}
for tick in range(24):
    time.sleep(5)
    with urllib.request.urlopen(
            urllib.request.Request(f'{BASE}/browser/session/{SID}', headers=H)) as r:
        result = json.loads(r.read())
    st = result.get('status', '?')
    f  = result.get('completed_fields_count', 0)
    print(f'   [{(tick+1)*5:3d}s] {st} | filled: {f}')
    if st in ('COMPLETED','WAITING_FOR_USER','FAILED','CANCELLED'): break

print()
print('='*62)
print('📊 RESULT')
print(f'  Status        : {result.get("status")}')
print(f'  Fields filled : {result.get("completed_fields_count",0)} / {result.get("total_fields_detected","?")}')
print(f'  URL           : {result.get("current_url")}')
if result.get('intervention_required'):
    itype = result.get('intervention_type', 'UNKNOWN')
    imsg  = result.get('intervention_message') or result.get('current_task')
    print(f'\n  ⚠️  HUMAN INPUT REQUIRED: {imsg}')
    print(f'     Intervention type : {itype}')
    if itype == 'MISSING_INFORMATION':
        print('\n  💡 The application needs personal/confidential information that the AI does not have.')
        print('     Enter the missing information below and submit to resume form filling:')
        print('     ---------------------------------------------------------------------------------')
        print('     # Example input:')
        print('     personal_answers = {"bvn": "22334455667", "passport_number": "A12345678"}')
        print('     cmd_data = json.dumps({"action": "submit_intervention", "intervention_data": personal_answers}).encode()')
        print('     req = urllib.request.Request(f"{BASE}/browser/{SID}/command", data=cmd_data, headers=H)')
        print('     with urllib.request.urlopen(req) as r: print("✅ Personal information submitted! Bot resumed.")')
    print(f'     Screenshot: {result.get("latest_screenshot_path")}')
    print('     (File is in your Google Drive screenshots folder)')
print('='*62)


In [ ]:
# ==============================================================
# CELL 6 — Discover a Grant by Scraping Any URL
#
# Paste any grant program page. The AI scout extracts:
# name, funder, amount, deadline, stage (GREEN/YELLOW/ORANGE/RED),
# eligible countries, questions, required documents.
# Saves to your database. Run Cell 5 to apply.
# ==============================================================

import urllib.request, urllib.parse, json

BASE = f'http://localhost:{API_PORT}/api/v1'

# ── ✏️  EDIT THIS ──────────────────────────────────────────────
GRANT_PAGE_URL = 'https://www.tonyelumelufoundation.org/entrepreneurship'
# ──────────────────────────────────────────────────────────────

login_data = urllib.parse.urlencode(
    {'username': 'demo@grantagent.ai', 'password': 'password123'}).encode()
with urllib.request.urlopen(
        urllib.request.Request(f'{BASE}/auth/login', data=login_data)) as r:
    TOKEN = json.loads(r.read())['access_token']
H = {'Authorization': f'Bearer {TOKEN}', 'Content-Type': 'application/json'}

print(f'🔍 Scraping: {GRANT_PAGE_URL}')
print('   Fetching page → Gemini AI → extracting details... (5-15 s)')
payload = json.dumps({'url': GRANT_PAGE_URL}).encode()
with urllib.request.urlopen(
        urllib.request.Request(f'{BASE}/grants/scrape', data=payload, headers=H)) as r:
    g = json.loads(r.read())

em = {'GREEN_IDEA':'🟢','GREEN_MVP':'🟢','YELLOW_VALIDATION':'🟡',
      'ORANGE_MVP':'🟠','RED_TRACTION':'🔴'}.get(g.get('stage_classification',''),'⚪')
mn, mx, cu = g.get('funding_amount_min'), g.get('funding_amount_max'), g.get('currency','USD')

print()
print('='*62)
print('🎯 GRANT DISCOVERED & SAVED')
print(f'  Name       : {g.get("grant_name")}')
print(f'  Funder     : {g.get("funder")}')
print(f'  Amount     : ${mn:,.0f} – ${mx:,.0f} {cu}' if mn and mx else '  Amount     : not specified')
print(f'  Deadline   : {g.get("deadline","not found")}')
print(f'  Stage      : {em} {g.get("stage_classification")}')
print(f'  Status     : {g.get("application_status")}')
print(f'  Verified   : {g.get("verification_status")}')
print(f'  Countries  : {g.get("eligible_countries",[])}')
print(f'  Grant ID   : {g.get("id")}')
if g.get('required_questions'):
    qs = g['required_questions']
    print(f'\n  Questions ({len(qs)}):')
    for i, q in enumerate(qs[:3], 1):
        print(f'    {i}. {q[:88]}...' if len(q)>88 else f'    {i}. {q}')
    if len(qs) > 3: print(f'    ... and {len(qs)-3} more')
print('\n  ✅ Saved. Run Cell 5 to apply.')
print('='*62)

In [ ]:
# ==============================================================
# CELL 7 — Deploy to Google Cloud Run (Always-on, 24/7)
#
# Colab sessions end after 12-24 h. Cloud Run gives Grant Agent
# a permanent public URL with no session limit.
#
# WHAT GETS DEPLOYED:
#   grant-agent-api    — FastAPI backend  (2 CPU / 2 GB / min 1)
#   grant-agent-worker — Playwright bot   (4 CPU / 4 GB / scale-0)
#   grant-agent-web    — Next.js frontend (1 CPU / 1 GB / min 1)
#   Cloud Scheduler    — daily grant scout at 02:00 UTC
#
# BEFORE RUNNING:
#   1. Create a GCP project at https://console.cloud.google.com
#   2. Enable billing on it
#   3. Set GCP_PROJECT_ID in Cell 2 to your real project ID
#   4. Create these secrets in Secret Manager (once, manually or
#      via deploy/gcp-secret-manager.sh from your local terminal):
#        grant-agent-db-url        — your database connection string
#        grant-agent-jwt-secret    — any random long string
#        grant-agent-gemini-key-1  — your Gemini API key 1
#        grant-agent-gemini-key-2  — key 2 (can be same as 1)
#        grant-agent-gemini-key-3  — key 3 (can be same as 1)
#
# WHAT HAPPENS WHEN YOU RUN:
#   Step 0: installs gcloud CLI (~2 min first time)
#   Step 1: opens browser auth — sign in with your Google account
#   Step 2: enables Cloud Run, Cloud Build, Artifact Registry APIs
#   Step 3: creates Docker image registry
#   Step 4: builds 3 Docker images via Cloud Build (~8-12 min)
#   Step 5-7: deploys 3 Cloud Run services (~3 min)
#   Step 8: creates Cloud Scheduler daily job
#   Total: ~15-20 minutes
#   At the end you get two permanent HTTPS URLs.
# ==============================================================

import subprocess, sys, os

REPO_DIR   = '/content/grant-agent'
PROJECT_ID = GCP_PROJECT_ID
REGION     = GCP_REGION
AR_REPO    = 'grant-agent-repo'

def run(cmd, **kw):
    print(f'\n$ {cmd[:100]}')
    r = subprocess.run(cmd, shell=True, **kw)
    if r.returncode not in (0, None):
        raise RuntimeError(f'Command failed (exit {r.returncode})')
    return r

# ── Step 0: gcloud CLI ───────────────────────────────────────────
print('='*62)
print('STEP 0: Verifying gcloud CLI')
print('='*62)
chk = subprocess.run('which gcloud', shell=True, capture_output=True)
if chk.returncode != 0:
    print('Installing gcloud (~300 MB, ~2 min)...')
    run('curl -sSL https://sdk.cloud.google.com | bash -s -- --disable-prompts --install-dir=/usr/local')
    os.environ['PATH'] += ':/usr/local/google-cloud-sdk/bin'
run('gcloud --version')

# ── Step 1: Auth ─────────────────────────────────────────────────
print()
print('='*62)
print('STEP 1: Authenticate with Google')
print('A URL will appear — open it, sign in, paste the code back.')
print('='*62)
run('gcloud auth login --no-launch-browser')
run(f'gcloud config set project {PROJECT_ID}')
run('gcloud config set core/disable_usage_reporting true')
print(f'✅ Authenticated. Project: {PROJECT_ID}')

# ── Step 2: Enable APIs ──────────────────────────────────────────
print()
print('='*62)
print('STEP 2: Enabling GCP APIs (~1 min)')
print('='*62)
run(f'gcloud services enable run.googleapis.com cloudbuild.googleapis.com '
    f'artifactregistry.googleapis.com secretmanager.googleapis.com '
    f'cloudscheduler.googleapis.com --project={PROJECT_ID}')
print('✅ APIs enabled')

# ── Step 3: Artifact Registry ────────────────────────────────────
print()
print('='*62)
print('STEP 3: Docker image registry')
print('='*62)
run(f'gcloud artifacts repositories create {AR_REPO} '
    f'--repository-format=docker --location={REGION} '
    f'--project={PROJECT_ID} 2>/dev/null || echo "Already exists"')
run(f'gcloud auth configure-docker {REGION}-docker.pkg.dev --quiet')
print('✅ Registry ready')

# ── Step 4: Build images ─────────────────────────────────────────
print()
print('='*62)
print('STEP 4: Building 3 Docker images via Cloud Build (~8-12 min)')
print('='*62)
IMG = f'{REGION}-docker.pkg.dev/{PROJECT_ID}/{AR_REPO}'
print('\n📦 Building API...')
run(f'gcloud builds submit --tag {IMG}/grant-agent-api:latest -f deploy/Dockerfile.api .', cwd=REPO_DIR)
print('\n📦 Building Worker (includes Chromium — largest image)...')
run(f'gcloud builds submit --tag {IMG}/grant-agent-worker:latest -f deploy/Dockerfile.worker .', cwd=REPO_DIR)
print('\n📦 Building Web (Next.js)...')
run(f'gcloud builds submit --tag {IMG}/grant-agent-web:latest -f deploy/Dockerfile.web .', cwd=REPO_DIR)
print('\n✅ All 3 images built')

# ── Step 5: Deploy API ───────────────────────────────────────────
print()
print('='*62)
print('STEP 5: Deploying FastAPI backend')
print('='*62)
run(f'''gcloud run deploy grant-agent-api \
    --image={IMG}/grant-agent-api:latest \
    --region={REGION} --platform=managed \
    --allow-unauthenticated \
    --cpu=2 --memory=2Gi \
    --min-instances=1 --max-instances=10 \
    --set-env-vars=ENVIRONMENT=production \
    --set-secrets=DATABASE_URL=grant-agent-db-url:latest,\
SECRET_KEY=grant-agent-jwt-secret:latest,\
GEMINI_API_KEY_1=grant-agent-gemini-key-1:latest,\
GEMINI_API_KEY_2=grant-agent-gemini-key-2:latest,\
GEMINI_API_KEY_3=grant-agent-gemini-key-3:latest \
    --project={PROJECT_ID}''')
API_URL = subprocess.run(
    f'gcloud run services describe grant-agent-api '
    f'--region={REGION} --format="value(status.url)" --project={PROJECT_ID}',
    shell=True, capture_output=True, text=True).stdout.strip()
print(f'✅ API: {API_URL}')

# ── Step 6: Deploy Worker ────────────────────────────────────────
print()
print('='*62)
print('STEP 6: Deploying Playwright worker (private, scales to 0)')
print('='*62)
run(f'''gcloud run deploy grant-agent-worker \
    --image={IMG}/grant-agent-worker:latest \
    --region={REGION} --platform=managed \
    --no-allow-unauthenticated \
    --cpu=4 --memory=4Gi --timeout=900 \
    --min-instances=0 --max-instances=5 \
    --set-secrets=DATABASE_URL=grant-agent-db-url:latest,\
SECRET_KEY=grant-agent-jwt-secret:latest \
    --project={PROJECT_ID}''')
print('✅ Worker deployed')

# ── Step 7: Deploy Web ───────────────────────────────────────────
print()
print('='*62)
print('STEP 7: Deploying Next.js frontend')
print('='*62)
run(f'''gcloud run deploy grant-agent-web \
    --image={IMG}/grant-agent-web:latest \
    --region={REGION} --platform=managed \
    --allow-unauthenticated \
    --cpu=1 --memory=1Gi \
    --min-instances=1 --max-instances=10 \
    --set-env-vars=NEXT_PUBLIC_API_URL={API_URL}/api/v1 \
    --project={PROJECT_ID}''')
WEB_URL = subprocess.run(
    f'gcloud run services describe grant-agent-web '
    f'--region={REGION} --format="value(status.url)" --project={PROJECT_ID}',
    shell=True, capture_output=True, text=True).stdout.strip()
print(f'✅ Frontend: {WEB_URL}')

# ── Step 8: Cloud Scheduler ──────────────────────────────────────
print()
print('='*62)
print('STEP 8: Daily grant scout (Cloud Scheduler, 02:00 UTC)')
print('='*62)
run(f'''gcloud scheduler jobs create http grant-agent-daily-scout \
    --schedule="0 2 * * *" \
    --uri={API_URL}/api/v1/automation/scout-run \
    --http-method=POST \
    --time-zone=Etc/UTC \
    --location={REGION} \
    --project={PROJECT_ID} 2>/dev/null || echo "Scheduler already exists"''')
print('✅ Daily scout scheduled')

# ── Summary ──────────────────────────────────────────────────────
print()
print('='*62)
print('🎉  DEPLOYED — Grant Agent is running 24/7!')
print('='*62)
print(f'  🌐  Web      : {WEB_URL}')
print(f'  📡  API      : {API_URL}')
print(f'  📖  API docs : {API_URL}/docs')
print(f'  ❤️   Health  : {API_URL}/health')
print()
print('  🔑  Login: demo@grantagent.ai / password123')
print(f'  ⏰  Daily scout: 02:00 UTC')
print(f'  ☁️   Project: {PROJECT_ID} / {REGION}')
print('='*62)